# Billups layer debugging

Use this notebook to inspect Bronze, Silver, and Gold with PySpark. Run `billups.pipeline` or the required individual stages before querying a layer. Keep outputs cleared before committing.

In [ ]:
import json
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "billups").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from pyspark.sql import functions as F
from billups.common import build_spark

spark = build_spark("billups-data-debugging")

In [ ]:
bronze_dir = project_root / "data" / "bronze"
silver_dir = project_root / "data" / "silver"
gold_dir = project_root / "data" / "gold"

print("Bronze:", bronze_dir.exists())
print("Silver:", silver_dir.exists())
print("Gold:", gold_dir.exists())

## Bronze

Bronze preserves the raw source grain in Parquet. Actions below are deliberately bounded.

In [ ]:
bronze_transactions = spark.read.parquet(str(bronze_dir / "historical_transactions"))
bronze_merchants = spark.read.parquet(str(bronze_dir / "merchants"))
bronze_transactions.printSchema()
bronze_transactions.limit(5).show(truncate=False)
bronze_merchants.select("merchant_id", "merchant_name").limit(5).show(truncate=False)

## Silver

Silver contains validated amounts and timestamps, normalized categories, and merchant-name enrichment.

In [ ]:
silver_transactions = spark.read.parquet(str(silver_dir / "transactions"))
silver_transactions.printSchema()
silver_transactions.select(
    "purchase_date", "purchase_amount", "merchant_id", "merchant_name", "category"
).limit(10).show(truncate=False)

with (silver_dir / "dq.json").open(encoding="utf-8") as stream:
    silver_dq = json.load(stream)
silver_dq

In [ ]:
silver_transactions.groupBy("authorized_flag").agg(
    F.count("*").alias("attempt_count"),
    F.sum("purchase_amount").alias("total_amount"),
).orderBy("authorized_flag").show()

## Gold

Gold tables contain the business answers consumed by the report and dashboard.

In [ ]:
gold_tables = sorted(path.name for path in gold_dir.iterdir() if path.is_dir())
gold_tables

In [ ]:
q1 = spark.read.parquet(str(gold_dir / "q1_top_merchants"))
q1.filter((F.col("year_month") == "2017-01") & (F.col("city_id") == 1)).orderBy("rank").show(5, truncate=False)

In [ ]:
q5_months = spark.read.parquet(str(gold_dir / "q5_months"))
q5_months.orderBy(F.desc("approved_amount_per_observed_day")).show(truncate=False)

## Stop Spark

Release the local Spark resources when debugging is complete.

In [ ]:
spark.stop()